Generate catalytic residue positions of CARs as pickle format based on alingment
---------------------------------------------------------------------------------

In [ ]:
import pickle
import alignment

msa = alignment.parse_msa("car_interpro_filtered.aln.fasta")

REF_ID = "A0AB38D5A4"

catalytic_map = {
    "cat_His": (301, "H"),
    "cat_Thr": (422, "T"),
    "A10_Lys": (618, "K"),
}

# Find PCP Ser column from the MSA, then translate to the reference's 1-based residue.
pcp = alignment.find_pcp_serine(msa, min_ser_freq=0.85, motif="GxxS")
if pcp is None:
    raise RuntimeError("PCP Ser not found; try relaxing thresholds.")

ref_aln = next(str(rec.seq).upper() for rec in msa if REF_ID in rec.id)
pcp_resid_1based = alignment.col_to_resid(ref_aln, pcp["col"]) + 1
catalytic_map["PCP_Ser"] = (pcp_resid_1based, "S")

result = alignment.map_catalytic_across_msa(
    msa,
    ref_id=REF_ID,
    catalytic_map=catalytic_map,
)

with open("car_catres_mapping.pkl", "wb") as f:
    car_catres_mapping = result["per_sequence"]
    pickle.dump(car_catres_mapping, f)

result["per_sequence"]

{'A0A010YLP6': {'cat_His': {'col': 629, 'aa': 'H', 'resid_1based': 296},
  'cat_Thr': {'col': 831, 'aa': 'T', 'resid_1based': 409},
  'A10_Lys': {'col': 1057, 'aa': 'K', 'resid_1based': 605},
  'PCP_Ser': {'col': 1144, 'aa': 'S', 'resid_1based': 678}},
 'A0A024JSK2': {'cat_His': {'col': 629, 'aa': 'H', 'resid_1based': 311},
  'cat_Thr': {'col': 831, 'aa': 'T', 'resid_1based': 432},
  'A10_Lys': {'col': 1057, 'aa': 'K', 'resid_1based': 628},
  'PCP_Ser': {'col': 1144, 'aa': 'S', 'resid_1based': 701}},
 'A0A024JWI0': {'cat_His': {'col': 629, 'aa': 'H', 'resid_1based': 294},
  'cat_Thr': {'col': 831, 'aa': 'T', 'resid_1based': 407},
  'A10_Lys': {'col': 1057, 'aa': 'K', 'resid_1based': 601},
  'PCP_Ser': {'col': 1144, 'aa': 'S', 'resid_1based': 674}},
 'A0A024JZE3': {'cat_His': {'col': 629, 'aa': 'H', 'resid_1based': 306},
  'cat_Thr': {'col': 831, 'aa': 'T', 'resid_1based': 425},
  'A10_Lys': {'col': 1057, 'aa': 'K', 'resid_1based': 614},
  'PCP_Ser': {'col': 1144, 'aa': 'S', 'resid_1bas

Obtain CAR A domain complex structures with both adipic acid and ATP, truncated by PCP-Ser residue point (S - 60 position)
----------------------------------------------------------------------------------------------------------

In [3]:
import os
import glob
import random
import pickle
import numpy as np
from Bio import PDB
from Bio.PDB.cealign import CEAligner
from Bio.PDB.Polypeptide import protein_letters_3to1
import pandas as pd

from tqdm.notebook import tqdm

# Config
TARGET_PDB = "../inputs/rosetta/carA_ref/macarA_holo_apa_atp_of3.pdb"

INPUT_DIR = "structures"
OUTPUT_DIR = "../inputs/rosetta/carA_holo_apa_atp_homologs/"
CATRES_MAPPING = "car_catres_mapping.pkl"
TRUNC_KEY = "PCP_Ser"
INCLUDE_CUT = False

LIGAND_MOL2 = [
    ("../inputs/rosetta/subs/apa.mol2", "LG0"),
    ("../inputs/rosetta/subs/atp.mol2", "LG1"),
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
parser = PDB.PDBParser(QUIET=True)
io = PDB.PDBIO()


def parse_mol2_to_hetatm(mol2_path: str, res_name: str = "LIG", chain_id: str = "Z") -> list:
    """Parse mol2 @<TRIPOS>ATOM section and convert to PDB HETATM format lines."""
    hetatm_lines = []
    in_atom_block = False

    with open(mol2_path, "r") as f:
        for line in f:
            if line.startswith("@<TRIPOS>ATOM"):
                in_atom_block = True
                continue
            if line.startswith("@<TRIPOS>") and in_atom_block:
                break
            if not in_atom_block or not line.strip():
                continue

            parts = line.split()
            atom_serial = int(parts[0])
            atom_name = parts[1][:4].ljust(4)
            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
            element = parts[5].split(".")[0][:2].upper()

            hetatm_line = (
                f"HETATM{atom_serial:5d} {atom_name} {res_name:3s} {chain_id}"
                f"{1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}{1.00:6.2f}{0.00:6.2f}          {element:>2s}\n"
            )
            hetatm_lines.append(hetatm_line)

    print(f"Ligand parsed: {len(hetatm_lines)} atoms from {os.path.basename(mol2_path)}")
    return hetatm_lines


def append_ligand_to_pdb(pdb_path: str, hetatm_lines: list):
    """Append ligand HETATM lines to an existing PDB file."""
    with open(pdb_path, "r") as f:
        lines = f.readlines()
    lines = [l for l in lines if not l.startswith("END")]
    with open(pdb_path, "w") as f:
        f.writelines(lines)
        f.writelines(hetatm_lines)
        f.write("END\n")


def trim_to_max_residue(struct, max_residue: int):
    """Detach residues outside 1-max_residue in-memory."""
    for model in struct:
        for chain in model:
            to_remove = [r.id for r in chain
                         if r.id[0] != " " or not (1 <= r.id[1] <= max_residue)]
            for rid in to_remove:
                chain.detach_child(rid)

def resolve_cut(seq_id: str, catres_map: dict, trunc_key: str,
                include_cut: bool) -> int:
    """
    Return 1-based cut residue (inclusive upper bound) for the given seq_id.
    Matches seq_id against ';'-split tokens of each map key.
    """
    for key in catres_map:
        if seq_id in key.split(";"):
            resid = catres_map[key][trunc_key]["resid_1based"]
            return resid if include_cut else resid - 1
    raise KeyError(f"{seq_id} not in catres_map")

def trim_and_align(mobile_path: str, target_struct, save_path: str, max_residue: int):
    mobile_struct = parser.get_structure("mobile", mobile_path)
    trim_to_max_residue(mobile_struct, max_residue)

    aligner = CEAligner()
    aligner.set_reference(target_struct)
    aligner.align(mobile_struct)

    io.set_structure(mobile_struct)
    io.save(save_path)
    return round(float(aligner.rms), 3)


# Load per-sequence catalytic-residue mapping.
with open(CATRES_MAPPING, "rb") as f:
    catres_map = pickle.load(f)

# Parse ligand(s)
if LIGAND_MOL2 is None:
    lig_paths = []
elif isinstance(LIGAND_MOL2, str):
    lig_paths = [LIGAND_MOL2]
else:
    lig_paths = list(LIGAND_MOL2)

chain_ids = "ZYXWVUT"
hetatm_lines = []
for i, (lig_path, lig_name) in enumerate(lig_paths):
    hetatm_lines.extend(parse_mol2_to_hetatm(lig_path, res_name=lig_name, chain_id=chain_ids[i]))

# Target is pre-trimmed; load as-is, no further cutting.
if TARGET_PDB is None:
    candidates = sorted(glob.glob(f"{INPUT_DIR}/car_*.pdb"))
    if not candidates:
        raise FileNotFoundError(f"No car_*.pdb files found in {INPUT_DIR}")
    TARGET_PDB = random.choice(candidates)
    print(f"Target not specified — randomly picked from {INPUT_DIR}: {TARGET_PDB}")

target_struct = parser.get_structure("target", TARGET_PDB)
print(f"Target loaded as-is: {TARGET_PDB}\n")

# Main loop
results = []
for pdb_path in tqdm(sorted(glob.glob(f"{INPUT_DIR}/car_*.pdb"))):
    filename = os.path.basename(pdb_path).replace("car_", "carA_")
    seq_id = os.path.splitext(os.path.basename(pdb_path))[0].replace("car_", "")
    save_path = os.path.join(OUTPUT_DIR, filename)

    try:
        cut = resolve_cut(seq_id, catres_map, TRUNC_KEY, INCLUDE_CUT) - 50
        rmsd = trim_and_align(pdb_path, target_struct, save_path, cut)
        if hetatm_lines:
            append_ligand_to_pdb(save_path, hetatm_lines)
        print(f"[SUCCESS]    {filename}  cut=1-{cut}  RMSD = {rmsd} Å")
        results.append({"file": filename, "cut": cut, "rmsd": rmsd})
    except Exception as e:
        print(f"[FAIL]  {filename} — {e}")

# Summary
summary = pd.DataFrame(results).sort_values("rmsd")
summary.to_csv("alignment_summary.csv", index=False)
print(f"\nAligned  : {len(results)} structures")
print(f"Mean RMSD: {summary['rmsd'].mean():.3f} Å")

Ligand parsed: 10 atoms from apa.mol2
Ligand parsed: 31 atoms from atp.mol2
Target loaded as-is: ../inputs/rosetta/carA_ref/macarA_holo_apa_atp_of3.pdb



  0%|          | 0/675 [00:00<?, ?it/s]

[SUCCESS]    carA_A0A010YLP6.pdb  cut=1-627  RMSD = 1.815 Å
[SUCCESS]    carA_A0A024JSK2.pdb  cut=1-650  RMSD = 2.437 Å
[SUCCESS]    carA_A0A024JWI0.pdb  cut=1-623  RMSD = 2.102 Å
[SUCCESS]    carA_A0A024JZE3.pdb  cut=1-636  RMSD = 1.899 Å
[SUCCESS]    carA_A0A024JZE8.pdb  cut=1-644  RMSD = 2.933 Å
[SUCCESS]    carA_A0A045K2P5.pdb  cut=1-628  RMSD = 2.153 Å
[SUCCESS]    carA_A0A051U3G5.pdb  cut=1-631  RMSD = 2.097 Å
[SUCCESS]    carA_A0A051UFG9.pdb  cut=1-650  RMSD = 3.453 Å
[SUCCESS]    carA_A0A064CG00.pdb  cut=1-627  RMSD = 2.251 Å
[SUCCESS]    carA_A0A0B1ZH49.pdb  cut=1-651  RMSD = 2.845 Å
[SUCCESS]    carA_A0A0B1ZIQ0.pdb  cut=1-599  RMSD = 1.766 Å
[SUCCESS]    carA_A0A0B8NM02.pdb  cut=1-641  RMSD = 2.137 Å
[SUCCESS]    carA_A0A0D1L7Y1.pdb  cut=1-632  RMSD = 2.232 Å
[SUCCESS]    carA_A0A0D1LMF4.pdb  cut=1-632  RMSD = 2.327 Å
[SUCCESS]    carA_A0A0D2EWW5.pdb  cut=1-624  RMSD = 2.288 Å
[SUCCESS]    carA_A0A0E4CLA0.pdb  cut=1-650  RMSD = 2.413 Å
[SUCCESS]    carA_A0A0E4GX90.pdb  cut=1-

Obtain CAR A domain complex structures with both adipic acid and AMP, truncated by PCP-Ser residue point (S - 60 position)
----------------------------------------------------------------------------------------------------------

In [7]:
import os
import glob
import random
import pickle
import numpy as np
from Bio import PDB
from Bio.PDB.cealign import CEAligner
from Bio.PDB.Polypeptide import protein_letters_3to1
import pandas as pd

from tqdm.notebook import tqdm

# Config
TARGET_PDB = "../inputs/rosetta/carA_ref/6OZ1.pdb"

INPUT_DIR = "structures"
OUTPUT_DIR = "../inputs/rosetta/carA_holo_adi_amp_homologs/"
CATRES_MAPPING = "car_catres_mapping.pkl"
TRUNC_KEY = "PCP_Ser"
INCLUDE_CUT = False

LIGAND_MOL2 = [
    ("../inputs/rosetta/subs/adi.mol2", "ADI"),
    ("../inputs/rosetta/subs/amp.mol2", "AMP")
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
parser = PDB.PDBParser(QUIET=True)
io = PDB.PDBIO()


def parse_mol2_to_hetatm(mol2_path: str, res_name: str = "LIG", chain_id: str = "Z") -> list:
    """Parse mol2 @<TRIPOS>ATOM section and convert to PDB HETATM format lines."""
    hetatm_lines = []
    in_atom_block = False

    with open(mol2_path, "r") as f:
        for line in f:
            if line.startswith("@<TRIPOS>ATOM"):
                in_atom_block = True
                continue
            if line.startswith("@<TRIPOS>") and in_atom_block:
                break
            if not in_atom_block or not line.strip():
                continue

            parts = line.split()
            atom_serial = int(parts[0])
            atom_name = parts[1][:4].ljust(4)
            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
            element = parts[5].split(".")[0][:2].upper()

            hetatm_line = (
                f"HETATM{atom_serial:5d} {atom_name} {res_name:3s} {chain_id}"
                f"{1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}{1.00:6.2f}{0.00:6.2f}          {element:>2s}\n"
            )
            hetatm_lines.append(hetatm_line)

    print(f"Ligand parsed: {len(hetatm_lines)} atoms from {os.path.basename(mol2_path)}")
    return hetatm_lines


def append_ligand_to_pdb(pdb_path: str, hetatm_lines: list):
    """Append ligand HETATM lines to an existing PDB file."""
    with open(pdb_path, "r") as f:
        lines = f.readlines()
    lines = [l for l in lines if not l.startswith("END")]
    with open(pdb_path, "w") as f:
        f.writelines(lines)
        f.writelines(hetatm_lines)
        f.write("END\n")


def trim_to_max_residue(struct, max_residue: int):
    """Detach residues outside 1-max_residue in-memory."""
    for model in struct:
        for chain in model:
            to_remove = [r.id for r in chain
                         if r.id[0] != " " or not (1 <= r.id[1] <= max_residue)]
            for rid in to_remove:
                chain.detach_child(rid)

def resolve_cut(seq_id: str, catres_map: dict, trunc_key: str,
                include_cut: bool) -> int:
    """
    Return 1-based cut residue (inclusive upper bound) for the given seq_id.
    Matches seq_id against ';'-split tokens of each map key.
    """
    for key in catres_map:
        if seq_id in key.split(";"):
            resid = catres_map[key][trunc_key]["resid_1based"]
            return resid if include_cut else resid - 1
    raise KeyError(f"{seq_id} not in catres_map")

def trim_and_align(mobile_path: str, target_struct, save_path: str, max_residue: int):
    mobile_struct = parser.get_structure("mobile", mobile_path)
    trim_to_max_residue(mobile_struct, max_residue)

    aligner = CEAligner()
    aligner.set_reference(target_struct)
    aligner.align(mobile_struct)

    io.set_structure(mobile_struct)
    io.save(save_path)
    return round(float(aligner.rms), 3)


# Load per-sequence catalytic-residue mapping.
with open(CATRES_MAPPING, "rb") as f:
    catres_map = pickle.load(f)

# Parse ligand(s)
if LIGAND_MOL2 is None:
    lig_paths = []
elif isinstance(LIGAND_MOL2, str):
    lig_paths = [LIGAND_MOL2]
else:
    lig_paths = list(LIGAND_MOL2)

chain_ids = "ZYXWVUT"
hetatm_lines = []
for i, (lig_path, lig_name) in enumerate(lig_paths):
    hetatm_lines.extend(parse_mol2_to_hetatm(lig_path, res_name=lig_name, chain_id=chain_ids[i]))

# Target is pre-trimmed; load as-is, no further cutting.
if TARGET_PDB is None:
    candidates = sorted(glob.glob(f"{INPUT_DIR}/car_*.pdb"))
    if not candidates:
        raise FileNotFoundError(f"No car_*.pdb files found in {INPUT_DIR}")
    TARGET_PDB = random.choice(candidates)
    print(f"Target not specified — randomly picked from {INPUT_DIR}: {TARGET_PDB}")

target_struct = parser.get_structure("target", TARGET_PDB)
print(f"Target loaded as-is: {TARGET_PDB}\n")

# Main loop
results = []
for pdb_path in tqdm(sorted(glob.glob(f"{INPUT_DIR}/car_*.pdb"))):
    filename = os.path.basename(pdb_path).replace("car_", "carA_")
    seq_id = os.path.splitext(os.path.basename(pdb_path))[0].replace("car_", "")
    save_path = os.path.join(OUTPUT_DIR, filename)

    try:
        cut = resolve_cut(seq_id, catres_map, TRUNC_KEY, INCLUDE_CUT) - 60
        rmsd = trim_and_align(pdb_path, target_struct, save_path, cut)
        if hetatm_lines:
            append_ligand_to_pdb(save_path, hetatm_lines)
        print(f"[SUCCESS]    {filename}  cut=1-{cut}  RMSD = {rmsd} Å")
        results.append({"file": filename, "cut": cut, "rmsd": rmsd})
    except Exception as e:
        print(f"[FAIL]  {filename} — {e}")

# Summary
summary = pd.DataFrame(results).sort_values("rmsd")
summary.to_csv("alignment_summary.csv", index=False)
print(f"\nAligned  : {len(results)} structures")
print(f"Mean RMSD: {summary['rmsd'].mean():.3f} Å")

Ligand parsed: 10 atoms from adi.mol2
Ligand parsed: 23 atoms from amp.mol2
Target loaded as-is: ../inputs/rosetta/carA_ref/6OZ1.pdb



  0%|          | 0/675 [00:00<?, ?it/s]

[SUCCESS]    carA_A0A010YLP6.pdb  cut=1-617  RMSD = 2.294 Å
[SUCCESS]    carA_A0A024JSK2.pdb  cut=1-640  RMSD = 2.131 Å
[SUCCESS]    carA_A0A024JWI0.pdb  cut=1-613  RMSD = 2.205 Å
[SUCCESS]    carA_A0A024JZE3.pdb  cut=1-626  RMSD = 1.99 Å
[SUCCESS]    carA_A0A024JZE8.pdb  cut=1-634  RMSD = 2.59 Å
[SUCCESS]    carA_A0A045K2P5.pdb  cut=1-618  RMSD = 2.246 Å
[SUCCESS]    carA_A0A051U3G5.pdb  cut=1-621  RMSD = 2.27 Å
[SUCCESS]    carA_A0A051UFG9.pdb  cut=1-640  RMSD = 2.744 Å
[SUCCESS]    carA_A0A064CG00.pdb  cut=1-617  RMSD = 2.335 Å
[SUCCESS]    carA_A0A0B1ZH49.pdb  cut=1-641  RMSD = 2.59 Å
[SUCCESS]    carA_A0A0B1ZIQ0.pdb  cut=1-589  RMSD = 1.927 Å
[SUCCESS]    carA_A0A0B8NM02.pdb  cut=1-631  RMSD = 3.054 Å
[SUCCESS]    carA_A0A0D1L7Y1.pdb  cut=1-622  RMSD = 2.538 Å
[SUCCESS]    carA_A0A0D1LMF4.pdb  cut=1-622  RMSD = 2.378 Å
[SUCCESS]    carA_A0A0D2EWW5.pdb  cut=1-614  RMSD = 2.369 Å
[SUCCESS]    carA_A0A0E4CLA0.pdb  cut=1-640  RMSD = 2.578 Å
[SUCCESS]    carA_A0A0E4GX90.pdb  cut=1-611 

Obtain CAR A domain complex structures with adipic acyl-AMP, truncated by PCP-Ser residue point (S - 60 position)
----------------------------------------------------------------------------------------------------------

In [8]:
import os
import glob
import random
import pickle
import numpy as np
from Bio import PDB
from Bio.PDB.cealign import CEAligner
from Bio.PDB.Polypeptide import protein_letters_3to1
import pandas as pd

from tqdm.notebook import tqdm

# Config
TARGET_PDB = "../inputs/rosetta/carA_ref/macarA_holo_apa_amp_of3.pdb"

INPUT_DIR = "structures"
OUTPUT_DIR = "../inputs/rosetta/carA_holo_apa_amp_homologs/"
CATRES_MAPPING = "car_catres_mapping.pkl"
TRUNC_KEY = "PCP_Ser"
INCLUDE_CUT = False

LIGAND_MOL2 = [
    ("../inputs/rosetta/subs/apa_amp.mol2", "LG2")
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
parser = PDB.PDBParser(QUIET=True)
io = PDB.PDBIO()


def parse_mol2_to_hetatm(mol2_path: str, res_name: str = "LIG", chain_id: str = "Z") -> list:
    """Parse mol2 @<TRIPOS>ATOM section and convert to PDB HETATM format lines."""
    hetatm_lines = []
    in_atom_block = False

    with open(mol2_path, "r") as f:
        for line in f:
            if line.startswith("@<TRIPOS>ATOM"):
                in_atom_block = True
                continue
            if line.startswith("@<TRIPOS>") and in_atom_block:
                break
            if not in_atom_block or not line.strip():
                continue

            parts = line.split()
            atom_serial = int(parts[0])
            atom_name = parts[1][:4].ljust(4)
            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
            element = parts[5].split(".")[0][:2].upper()

            hetatm_line = (
                f"HETATM{atom_serial:5d} {atom_name} {res_name:3s} {chain_id}"
                f"{1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}{1.00:6.2f}{0.00:6.2f}          {element:>2s}\n"
            )
            hetatm_lines.append(hetatm_line)

    print(f"Ligand parsed: {len(hetatm_lines)} atoms from {os.path.basename(mol2_path)}")
    return hetatm_lines


def append_ligand_to_pdb(pdb_path: str, hetatm_lines: list):
    """Append ligand HETATM lines to an existing PDB file."""
    with open(pdb_path, "r") as f:
        lines = f.readlines()
    lines = [l for l in lines if not l.startswith("END")]
    with open(pdb_path, "w") as f:
        f.writelines(lines)
        f.writelines(hetatm_lines)
        f.write("END\n")


def trim_to_max_residue(struct, max_residue: int):
    """Detach residues outside 1-max_residue in-memory."""
    for model in struct:
        for chain in model:
            to_remove = [r.id for r in chain
                         if r.id[0] != " " or not (1 <= r.id[1] <= max_residue)]
            for rid in to_remove:
                chain.detach_child(rid)

def resolve_cut(seq_id: str, catres_map: dict, trunc_key: str,
                include_cut: bool) -> int:
    """
    Return 1-based cut residue (inclusive upper bound) for the given seq_id.
    Matches seq_id against ';'-split tokens of each map key.
    """
    for key in catres_map:
        if seq_id in key.split(";"):
            resid = catres_map[key][trunc_key]["resid_1based"]
            return resid if include_cut else resid - 1
    raise KeyError(f"{seq_id} not in catres_map")

def trim_and_align(mobile_path: str, target_struct, save_path: str, max_residue: int):
    mobile_struct = parser.get_structure("mobile", mobile_path)
    trim_to_max_residue(mobile_struct, max_residue)

    aligner = CEAligner()
    aligner.set_reference(target_struct)
    aligner.align(mobile_struct)

    io.set_structure(mobile_struct)
    io.save(save_path)
    return round(float(aligner.rms), 3)


# Load per-sequence catalytic-residue mapping.
with open(CATRES_MAPPING, "rb") as f:
    catres_map = pickle.load(f)

# Parse ligand(s)
if LIGAND_MOL2 is None:
    lig_paths = []
elif isinstance(LIGAND_MOL2, str):
    lig_paths = [LIGAND_MOL2]
else:
    lig_paths = list(LIGAND_MOL2)

chain_ids = "ZYXWVUT"
hetatm_lines = []
for i, (lig_path, lig_name) in enumerate(lig_paths):
    hetatm_lines.extend(parse_mol2_to_hetatm(lig_path, res_name=lig_name, chain_id=chain_ids[i]))

# Target is pre-trimmed; load as-is, no further cutting.
if TARGET_PDB is None:
    candidates = sorted(glob.glob(f"{INPUT_DIR}/car_*.pdb"))
    if not candidates:
        raise FileNotFoundError(f"No car_*.pdb files found in {INPUT_DIR}")
    TARGET_PDB = random.choice(candidates)
    print(f"Target not specified — randomly picked from {INPUT_DIR}: {TARGET_PDB}")

target_struct = parser.get_structure("target", TARGET_PDB)
print(f"Target loaded as-is: {TARGET_PDB}\n")

# Main loop
results = []
for pdb_path in tqdm(sorted(glob.glob(f"{INPUT_DIR}/car_*.pdb"))):
    filename = os.path.basename(pdb_path).replace("car_", "carA_")
    seq_id = os.path.splitext(os.path.basename(pdb_path))[0].replace("car_", "")
    save_path = os.path.join(OUTPUT_DIR, filename)

    try:
        cut = resolve_cut(seq_id, catres_map, TRUNC_KEY, INCLUDE_CUT) - 60
        rmsd = trim_and_align(pdb_path, target_struct, save_path, cut)
        if hetatm_lines:
            append_ligand_to_pdb(save_path, hetatm_lines)
        print(f"[SUCCESS]    {filename}  cut=1-{cut}  RMSD = {rmsd} Å")
        results.append({"file": filename, "cut": cut, "rmsd": rmsd})
    except Exception as e:
        print(f"[FAIL]  {filename} — {e}")

# Summary
summary = pd.DataFrame(results).sort_values("rmsd")
summary.to_csv("alignment_summary.csv", index=False)
print(f"\nAligned  : {len(results)} structures")
print(f"Mean RMSD: {summary['rmsd'].mean():.3f} Å")

Ligand parsed: 32 atoms from apa_amp.mol2
Target loaded as-is: ../inputs/rosetta/carA_ref/macarA_holo_apa_amp_of3.pdb



  0%|          | 0/675 [00:00<?, ?it/s]

[SUCCESS]    carA_A0A010YLP6.pdb  cut=1-617  RMSD = 1.757 Å
[SUCCESS]    carA_A0A024JSK2.pdb  cut=1-640  RMSD = 4.243 Å
[SUCCESS]    carA_A0A024JWI0.pdb  cut=1-613  RMSD = 2.035 Å
[SUCCESS]    carA_A0A024JZE3.pdb  cut=1-626  RMSD = 2.12 Å
[SUCCESS]    carA_A0A024JZE8.pdb  cut=1-634  RMSD = 3.052 Å
[SUCCESS]    carA_A0A045K2P5.pdb  cut=1-618  RMSD = 2.186 Å
[SUCCESS]    carA_A0A051U3G5.pdb  cut=1-621  RMSD = 2.177 Å
[SUCCESS]    carA_A0A051UFG9.pdb  cut=1-640  RMSD = 3.565 Å
[SUCCESS]    carA_A0A064CG00.pdb  cut=1-617  RMSD = 2.297 Å
[SUCCESS]    carA_A0A0B1ZH49.pdb  cut=1-641  RMSD = 2.842 Å
[SUCCESS]    carA_A0A0B1ZIQ0.pdb  cut=1-589  RMSD = 1.828 Å
[SUCCESS]    carA_A0A0B8NM02.pdb  cut=1-631  RMSD = 2.867 Å
[SUCCESS]    carA_A0A0D1L7Y1.pdb  cut=1-622  RMSD = 2.278 Å
[SUCCESS]    carA_A0A0D1LMF4.pdb  cut=1-622  RMSD = 2.5 Å
[SUCCESS]    carA_A0A0D2EWW5.pdb  cut=1-614  RMSD = 2.288 Å
[SUCCESS]    carA_A0A0E4CLA0.pdb  cut=1-640  RMSD = 2.511 Å
[SUCCESS]    carA_A0A0E4GX90.pdb  cut=1-611

Obtain CAR A domain complex structures with both 6-aminocaproic acid (6-ACA) and AMP, truncated by PCP-Ser residue point (S - 60 position)
----------------------------------------------------------------------------------------------------------

In [1]:
import os
import glob
import random
import pickle
import numpy as np
from Bio import PDB
from Bio.PDB.cealign import CEAligner
from Bio.PDB.Polypeptide import protein_letters_3to1
import pandas as pd

from tqdm.notebook import tqdm

# Config
TARGET_PDB = "../inputs/rosetta/carA_ref/6OZ1.pdb"

INPUT_DIR = "structures"
OUTPUT_DIR = "../inputs/rosetta/carA_holo_6aca_amp_homologs/"
CATRES_MAPPING = "car_catres_mapping.pkl"
TRUNC_KEY = "PCP_Ser"
INCLUDE_CUT = False

LIGAND_MOL2 = [
    ("../inputs/rosetta/subs/6aca.mol2", "ADI"),
    ("../inputs/rosetta/subs/amp.mol2", "AMP")
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
parser = PDB.PDBParser(QUIET=True)
io = PDB.PDBIO()


def parse_mol2_to_hetatm(mol2_path: str, res_name: str = "LIG", chain_id: str = "Z") -> list:
    """Parse mol2 @<TRIPOS>ATOM section and convert to PDB HETATM format lines."""
    hetatm_lines = []
    in_atom_block = False

    with open(mol2_path, "r") as f:
        for line in f:
            if line.startswith("@<TRIPOS>ATOM"):
                in_atom_block = True
                continue
            if line.startswith("@<TRIPOS>") and in_atom_block:
                break
            if not in_atom_block or not line.strip():
                continue

            parts = line.split()
            atom_serial = int(parts[0])
            atom_name = parts[1][:4].ljust(4)
            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
            element = parts[5].split(".")[0][:2].upper()

            hetatm_line = (
                f"HETATM{atom_serial:5d} {atom_name} {res_name:3s} {chain_id}"
                f"{1:4d}    {x:8.3f}{y:8.3f}{z:8.3f}{1.00:6.2f}{0.00:6.2f}          {element:>2s}\n"
            )
            hetatm_lines.append(hetatm_line)

    print(f"Ligand parsed: {len(hetatm_lines)} atoms from {os.path.basename(mol2_path)}")
    return hetatm_lines


def append_ligand_to_pdb(pdb_path: str, hetatm_lines: list):
    """Append ligand HETATM lines to an existing PDB file."""
    with open(pdb_path, "r") as f:
        lines = f.readlines()
    lines = [l for l in lines if not l.startswith("END")]
    with open(pdb_path, "w") as f:
        f.writelines(lines)
        f.writelines(hetatm_lines)
        f.write("END\n")


def trim_to_max_residue(struct, max_residue: int):
    """Detach residues outside 1-max_residue in-memory."""
    for model in struct:
        for chain in model:
            to_remove = [r.id for r in chain
                         if r.id[0] != " " or not (1 <= r.id[1] <= max_residue)]
            for rid in to_remove:
                chain.detach_child(rid)

def resolve_cut(seq_id: str, catres_map: dict, trunc_key: str,
                include_cut: bool) -> int:
    """
    Return 1-based cut residue (inclusive upper bound) for the given seq_id.
    Matches seq_id against ';'-split tokens of each map key.
    """
    for key in catres_map:
        if seq_id in key.split(";"):
            resid = catres_map[key][trunc_key]["resid_1based"]
            return resid if include_cut else resid - 1
    raise KeyError(f"{seq_id} not in catres_map")

def trim_and_align(mobile_path: str, target_struct, save_path: str, max_residue: int):
    mobile_struct = parser.get_structure("mobile", mobile_path)
    trim_to_max_residue(mobile_struct, max_residue)

    aligner = CEAligner()
    aligner.set_reference(target_struct)
    aligner.align(mobile_struct)

    io.set_structure(mobile_struct)
    io.save(save_path)
    return round(float(aligner.rms), 3)


# Load per-sequence catalytic-residue mapping.
with open(CATRES_MAPPING, "rb") as f:
    catres_map = pickle.load(f)

# Parse ligand(s)
if LIGAND_MOL2 is None:
    lig_paths = []
elif isinstance(LIGAND_MOL2, str):
    lig_paths = [LIGAND_MOL2]
else:
    lig_paths = list(LIGAND_MOL2)

chain_ids = "ZYXWVUT"
hetatm_lines = []
for i, (lig_path, lig_name) in enumerate(lig_paths):
    hetatm_lines.extend(parse_mol2_to_hetatm(lig_path, res_name=lig_name, chain_id=chain_ids[i]))

# Target is pre-trimmed; load as-is, no further cutting.
if TARGET_PDB is None:
    candidates = sorted(glob.glob(f"{INPUT_DIR}/car_*.pdb"))
    if not candidates:
        raise FileNotFoundError(f"No car_*.pdb files found in {INPUT_DIR}")
    TARGET_PDB = random.choice(candidates)
    print(f"Target not specified — randomly picked from {INPUT_DIR}: {TARGET_PDB}")

target_struct = parser.get_structure("target", TARGET_PDB)
print(f"Target loaded as-is: {TARGET_PDB}\n")

# Main loop
results = []
for pdb_path in tqdm(sorted(glob.glob(f"{INPUT_DIR}/car_*.pdb"))):
    filename = os.path.basename(pdb_path).replace("car_", "carA_")
    seq_id = os.path.splitext(os.path.basename(pdb_path))[0].replace("car_", "")
    save_path = os.path.join(OUTPUT_DIR, filename)

    try:
        cut = resolve_cut(seq_id, catres_map, TRUNC_KEY, INCLUDE_CUT) - 60
        rmsd = trim_and_align(pdb_path, target_struct, save_path, cut)
        if hetatm_lines:
            append_ligand_to_pdb(save_path, hetatm_lines)
        print(f"[SUCCESS]    {filename}  cut=1-{cut}  RMSD = {rmsd} Å")
        results.append({"file": filename, "cut": cut, "rmsd": rmsd})
    except Exception as e:
        print(f"[FAIL]  {filename} — {e}")

# Summary
summary = pd.DataFrame(results).sort_values("rmsd")
summary.to_csv("alignment_summary.csv", index=False)
print(f"\nAligned  : {len(results)} structures")
print(f"Mean RMSD: {summary['rmsd'].mean():.3f} Å")

Ligand parsed: 9 atoms from 6aca.mol2
Ligand parsed: 23 atoms from amp.mol2
Target loaded as-is: ../inputs/rosetta/carA_ref/6OZ1.pdb



  0%|          | 0/675 [00:00<?, ?it/s]

[SUCCESS]    carA_A0A010YLP6.pdb  cut=1-617  RMSD = 2.294 Å
[SUCCESS]    carA_A0A024JSK2.pdb  cut=1-640  RMSD = 2.131 Å
[SUCCESS]    carA_A0A024JWI0.pdb  cut=1-613  RMSD = 2.205 Å
[SUCCESS]    carA_A0A024JZE3.pdb  cut=1-626  RMSD = 1.99 Å
[SUCCESS]    carA_A0A024JZE8.pdb  cut=1-634  RMSD = 2.59 Å
[SUCCESS]    carA_A0A045K2P5.pdb  cut=1-618  RMSD = 2.246 Å
[SUCCESS]    carA_A0A051U3G5.pdb  cut=1-621  RMSD = 2.27 Å
[SUCCESS]    carA_A0A051UFG9.pdb  cut=1-640  RMSD = 2.744 Å
[SUCCESS]    carA_A0A064CG00.pdb  cut=1-617  RMSD = 2.335 Å
[SUCCESS]    carA_A0A0B1ZH49.pdb  cut=1-641  RMSD = 2.59 Å
[SUCCESS]    carA_A0A0B1ZIQ0.pdb  cut=1-589  RMSD = 1.927 Å
[SUCCESS]    carA_A0A0B8NM02.pdb  cut=1-631  RMSD = 3.054 Å
[SUCCESS]    carA_A0A0D1L7Y1.pdb  cut=1-622  RMSD = 2.538 Å
[SUCCESS]    carA_A0A0D1LMF4.pdb  cut=1-622  RMSD = 2.378 Å
[SUCCESS]    carA_A0A0D2EWW5.pdb  cut=1-614  RMSD = 2.369 Å
[SUCCESS]    carA_A0A0E4CLA0.pdb  cut=1-640  RMSD = 2.578 Å
[SUCCESS]    carA_A0A0E4GX90.pdb  cut=1-611 